Hello there, this is Reforge for colab. Adjust the models, extension by yourself :D

Make sure to check the code and adjust what you need. I'm not good at optimizing it!

I don't know if this work with Controlnet. I don't know how to do that, and I have never tried that thing before so no knowledge of using it.

Good luck!



In [ ]:
!git clone https://github.com/Panchovix/stable-diffusion-webui-reForge.git
%cd /content/stable-diffusion-webui-reForge


In [ ]:
# --- CONFIGURATION ---
# Make sure to adjust these depend on what you want
CIVITAI_API_KEY = ""            # You need API key to download something from CivitAI
HF_TOKEN = ""                   # Only need when model require the token

# TOGGLES:
LINK_DRIVE_LORA = True          # True = Show existing LoRAs from your Drive in the UI
SAVE_DOWNLOADS_TO_DRIVE = False # True = New downloads save to Drive; False = Save to local UI only

CHECKPOINT_URLS = [
    "https://huggingface.co/Manityro/Vermilion/blob/main/Vermilion-V3.1.safetensors",
    "https://huggingface.co/justTNP/MonsterCoffeeCKPTS/blob/main/MonsterCoffee5HE-ChenkinNoobv01_v1.safetensors"
]
VAE_URLS = ["https://huggingface.co/Anzhc/Anzhcs-VAEs/resolve/main/AAA%20Anime%20VAE%20SDXL%20v2.safetensors"]
LORA_URLS = ["https://civitai.com/api/download/models/2639596?type=Model&format=SafeTensor"]

DRIVE_BASE_PATH = "/content/drive/MyDrive/reForge_Data"
WebUI = "/content/stable-diffusion-webui-reForge"

# --- INTERNAL SETUP ---
import os
from google.colab import drive
from urllib.parse import unquote

# 1. Mount Drive
if not os.path.exists("/content/drive"): drive.mount('/content/drive')

# 2. Setup Paths
CKPT_DIR = f"{WebUI}/models/Stable-diffusion"
VAE_DIR = f"{WebUI}/models/VAE"
LOCAL_LORA = f"{WebUI}/models/Lora"
GD_LORA = f"{DRIVE_BASE_PATH}/models/Lora"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(VAE_DIR, exist_ok=True)
os.makedirs(GD_LORA, exist_ok=True)

# 3. Install aria2
if not os.path.exists("/usr/bin/aria2c"):
    print("📦 Installing aria2...")
    !apt-get -y install -qq aria2

# 4. Export LoRA from Drive
if LINK_DRIVE_LORA:
    if os.path.exists(LOCAL_LORA) and not os.path.islink(LOCAL_LORA):
        !rm -rf "{LOCAL_LORA}"
    if not os.path.islink(LOCAL_LORA):
        !ln -s "{GD_LORA}" "{LOCAL_LORA}"
    print("🔗 Drive LoRAs linked to UI.")
else:
    if os.path.islink(LOCAL_LORA):
      !rm "{LOCAL_LORA}"
    os.makedirs(LOCAL_LORA, exist_ok=True)
    print("📂 Using local LoRA storage only.")

# 5. Determine Download Destination
FINAL_LORA_PATH = GD_LORA if SAVE_DOWNLOADS_TO_DRIVE else f"{WebUI}/models/Lora"

# DOWNLOAD
def parse_links(url_list):
    return [u.replace('/blob/', '/resolve/') for u in url_list]

def optimized_download(links, target_path, site_name="Model"):
    if not links: return
    for url in links:
        filename = unquote(url.split('/')[-1].split('?')[0])
        if filename.isdigit(): filename = f"model_{filename}.safetensors"

        header = f'--header="Authorization: Bearer {HF_TOKEN}"' if "huggingface.co" in url and HF_TOKEN else ""
        final_url = f"{url}{'&' if '?' in url else '?'}token={CIVITAI_API_KEY}" if "civitai.com" in url else url

        print(f"📥 Downloading {site_name}: {filename}")
        !aria2c --console-log-level=error -c -x 16 -s 16 "{final_url}" {header} -d "{target_path}" -o "{filename}"

# EXECUTION
print("--- Starting Downloads ---")
optimized_download(parse_links(CHECKPOINT_URLS), CKPT_DIR, "Checkpoint")
optimized_download(parse_links(VAE_URLS), VAE_DIR, "VAE")
optimized_download(parse_links(LORA_URLS), FINAL_LORA_PATH, "LoRA")

print("\n✅ SETUP COMPLETE!")

In [ ]:
# Add your custom extension here!

import os

# Add extension URLs here
extensions = [
    "https://github.com/Bing-su/adetailer",
    "https://github.com/DominikDoom/a1111-sd-webui-tagcomplete",
    "https://github.com/hako-mikan/sd-webui-regional-prompter",
    "https://github.com/Haoming02/sd-forge-couple",
    "https://github.com/Anzhc/ArcEnCiel-Extension-for-WebUI",
]

# Add ADetailer
adet_models = [
    "https://huggingface.co/Anzhc/Anzhcs_YOLOs/resolve/main/Anzhc%20Breasts%20Seg%20v1%201024m.pt",
    "https://huggingface.co/Anzhc/Anzhcs_YOLOs/resolve/main/Anzhc%20Eyes%20-seg-hd.pt",
    "https://huggingface.co/Anzhc/Anzhcs_YOLOs/resolve/main/Anzhc%20Face%20-seg.pt",
    "https://huggingface.co/Anzhc/Anzhcs_YOLOs/resolve/main/Anzhc%20Face%20seg%201024%20v2%20y8n.pt",
    "https://huggingface.co/Anzhc/Anzhcs_YOLOs/resolve/main/Anzhc%20HeadHair%20seg%20y8m.pt",
    "https://huggingface.co/Anzhc/Anzhcs_YOLOs/resolve/main/Anzhc%20Manga%20Panels%20-seg.pt",
    "https://huggingface.co/Anzhc/Anzhcs_YOLOs/resolve/main/Anzhcs%20ManFace%20v02%201024%20y8n.pt",
    "https://huggingface.co/Anzhc/Anzhcs_YOLOs/resolve/main/Anzhcs%20WomanFace%20v05%201024%20y8n.pt",
    "https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8n.pt"
]

# Add Upscaler
upscalers = [
    "https://huggingface.co/embed/upscale/resolve/main/4x-UltraSharp.pth"
]

# Execution
WebUI = "/content/stable-diffusion-webui-reForge"

print("Cloning Extensions...")
for repo in extensions:
    repo_name = repo.split("/")[-1]
    target_path = f"{WebUI}/extensions/{repo_name}"
    if not os.path.exists(target_path):
        !git clone {repo} {target_path}
    else:
        print(f"Extension {repo_name} already exists.")

# Put models into correct path
adet_path = f"{WebUI}/models/adetailer"
esrgan_path = f"{WebUI}/models/ESRGAN"

os.makedirs(adet_path, exist_ok=True)
os.makedirs(esrgan_path, exist_ok=True)

print("Downloading ADetailer Models...")
for url in adet_models:
    filename = url.split("/")[-1].replace("%20", " ")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{adet_path}" -o "{filename}"

print("Downloading Upscalers...")
for url in upscalers:
    filename = url.split("/")[-1].replace("%20", " ")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{esrgan_path}" -o "{filename}"

print("✅ All extensions and models updated!")


In [ ]:
# @title <b><font color='orange'>Run only for ArcEnCiel-Extension-for-WebUI</font></b> {"display-mode":"form"}
# Separate toggles for storage management
SAVE_LORA_TO_DRIVE = False #@param {type:"boolean"}
SAVE_CHECKPOINT_TO_DRIVE = False #@param {type:"boolean"}

#CHANGE THE DOWNLOAD PATH
# Define Paths
save_paths_file = "/content/stable-diffusion-webui-reForge/extensions/ArcEnCiel-Extension-for-WebUI/save_paths.txt"
WebUI = "/content/stable-diffusion-webui-reForge"
Drive_Base = "/content/drive/MyDrive/reForge_Data"

# 1. Determine LORA Path
if SAVE_LORA_TO_DRIVE:
    LORA_PATH = f"{Drive_Base}/models/Lora"
    os.makedirs(LORA_PATH, exist_ok=True)
else:
    LORA_PATH = f"{WebUI}/models/Lora"

# 2. Determine CHECKPOINT Path
if SAVE_CHECKPOINT_TO_DRIVE:
    CKPT_PATH = f"{Drive_Base}/models/Stable-diffusion"
    os.makedirs(CKPT_PATH, exist_ok=True)
else:
    CKPT_PATH = f"{WebUI}/models/Stable-diffusion"

# 3. Construct and Write the Config
new_content = f"""LORA={LORA_PATH}
CHECKPOINT={CKPT_PATH}"""

if os.path.exists(os.path.dirname(save_paths_file)):
    with open(save_paths_file, "w") as f:
        f.write(new_content)

#ADD MORE SEARCH MODELS:
file_path = "/content/stable-diffusion-webui-reForge/extensions/ArcEnCiel-Extension-for-WebUI/scripts/arcenciel_gui.py"

# Read the original file
with open(file_path, "r") as f:
    lines = f.readlines()

# Define the new base model choices you requested
new_choices = [
    "Any", "Illustrious", "Illustrious 0.1", "NoobAI Eps", "NoobAI V-Pred",
    "NoobAI Flux2V", "Lumina", "Chenkin RF", "Chroma", "Hunyuan Framepack",
    "WAN 2.1", "Pony", "Flux.1 D", "Flux.1 S", "SDXL 1.0", "SD1.5"
]

start_line = -1
end_line = -1

for i, line in enumerate(lines):
    if 'label="Base Model",' in line:
        start_line = i + 1
    if start_line != -1 and 'choices=[' in lines[i]:
        for j in range(i, i + 10):
            if ']' in lines[j]:
                end_line = j
                break
        if end_line != -1:
            # Replace the lines between the brackets with our new list
            formatted_choices = ",\n".join([f'"{c}"' for c in new_choices])
            lines[i:end_line+1] = [f'choices=[\n {formatted_choices}\n],\n']
            break

# Write the changes back to the file
with open(file_path, "w") as f:
    f.writelines(lines)

print(f"✅ Successfully updated {file_path} with newer model types.")
print("--- 📂 ArcEnCiel Configuration Updated ---")
print(f"✨ LoRAs: {'🚀 Google Drive' if SAVE_LORA_TO_DRIVE else '💻 Local (Temp)'}")
print(f"✨ Checkpoints: {'🚀 Google Drive' if SAVE_CHECKPOINT_TO_DRIVE else '💻 Local (Temp)'}")
print(f"📍 Config saved to: {save_paths_file}")

In [ ]:
%cd /content/stable-diffusion-webui-reForge
!COMMANDLINE_ARGS="--theme dark --xformers --cuda-stream --share --enable-insecure-extension-access" REQS_FILE="requirements_versions.txt" python launch.py